In [ ]:
from pathlib import Path
import pandas as pd

csv_path = Path('data/raw/ld50-smiles-descriptors-dataset.csv')
if not csv_path.is_file():
    csv_path = Path('../data/raw/ld50-smiles-descriptors-dataset.csv')

df = pd.read_csv(csv_path)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## Clean and prepare the data

Work on a copy of `df`. `LD50` is already stored as −log10(LD50 mol/kg), so it is not transformed. Only missing target values and exact duplicate rows are removed.

### 1. Create a working copy

In [ ]:
# Copy the data so the original stays unchanged.
clean_df = df.copy()
# Record the starting number of rows.
initial_rows = len(clean_df)
# Choose the two columns to check.
main_columns = ['LD50', 'NumHAcceptors']

### 2. Check missing target values

In [ ]:
# Count missing values in each selected column.
missing_values = clean_df[main_columns].isnull().sum()
# Display the missing-value counts.
missing_values

### 3. Remove rows missing LD50 or NumHAcceptors

In [ ]:
# Remove rows missing LD50 or NumHAcceptors.
clean_df = clean_df.dropna(subset=['LD50', 'NumHAcceptors'])
# Calculate how many rows were removed.
missing_rows_removed = initial_rows - len(clean_df)
# Print the number of removed rows.
print(f'Rows removed for missing target values: {missing_rows_removed}')
# Check that neither selected column has missing values left.
assert not clean_df[main_columns].isnull().any().any()
# Display the remaining missing-value counts.
clean_df[main_columns].isnull().sum()

### 4. Check exact duplicate rows

In [ ]:
# Count rows that exactly repeat an earlier row.
exact_duplicate_rows = int(clean_df.duplicated().sum())
# Display the number of exact duplicate rows.
exact_duplicate_rows

### 5. Check duplicate SMILES separately

Repeated SMILES may represent separate toxicity records, so SMILES alone is not used to remove rows.

In [ ]:
# Count SMILES entries that repeat an earlier value.
duplicate_smiles_before = int(clean_df['SMILES'].duplicated().sum())
# Display the number of repeated SMILES entries.
duplicate_smiles_before

### 6. Remove exact duplicates

Keep the first occurrence only when every column is identical. Retain other records that share a SMILES value.

In [ ]:
# Record the row count before removing duplicates.
rows_before_deduplication = len(clean_df)
# Run the removal only if exact duplicates exist.
if exact_duplicate_rows > 0:
    # Keep the first copy of each identical row.
    clean_df = clean_df.drop_duplicates()
# Calculate how many duplicate rows were removed.
exact_rows_removed = rows_before_deduplication - len(clean_df)
# Check that no exact duplicate rows remain.
assert not clean_df.duplicated().any()
# Print the number of duplicate rows removed.
print(f'Exact duplicate rows removed: {exact_rows_removed}')
# Print how many repeated SMILES entries remain.
print(f'Remaining repeated SMILES values: {clean_df["SMILES"].duplicated().sum()}')

### 7. Confirm numeric values

Check that both target columns are numeric and finite. Stop for review if validation fails instead of silently changing values.

In [ ]:
# Import NumPy to check for missing or infinite numbers.
import numpy as np
# Import a helper that checks whether a data type is numeric.
from pandas.api.types import is_numeric_dtype

# Check whether each selected column has a numeric data type.
numeric_columns = clean_df[main_columns].dtypes.apply(is_numeric_dtype)
# Stop with an error if either column is not numeric.
assert numeric_columns.all(), 'LD50 and NumHAcceptors must both be numeric.'
# Check that both columns contain no missing or infinite numbers.
assert np.isfinite(clean_df[main_columns]).all().all(), 'Review non-finite target values.'
# Display the data type of each selected column.
clean_df[main_columns].dtypes

### 8. Validate hydrogen-bond acceptor counts

In [ ]:
# Select the hydrogen-bond acceptor counts.
acceptor_counts = clean_df['NumHAcceptors']
# Mark counts that are whole numbers and zero or greater.
valid_acceptor_counts = acceptor_counts.ge(0) & acceptor_counts.mod(1).eq(0)
# Print how many counts failed these checks.
print(f'Invalid acceptor counts: {(~valid_acceptor_counts).sum()}')
# Check that every acceptor count is valid.
assert valid_acceptor_counts.all(), 'Review negative or fractional NumHAcceptors values.'
# Show how often each count occurs, from smallest to largest.
clean_df['NumHAcceptors'].value_counts().sort_index()

### 9. Inspect extreme values

Inspect ranges, tail percentiles, and the five lowest and highest records for each target column. Unusual values are retained; these checks do not establish that an extreme value is an error.

In [ ]:
# Show summary statistics, including the 1st, 50th, and 99th percentiles.
clean_df[main_columns].describe(percentiles=[0.01, 0.5, 0.99])

In [ ]:
# Import the function used to show tables in the notebook.
from IPython.display import display

# Choose which columns to show for each record.
inspection_columns = ['Name', 'SMILES', 'LD50', 'NumHAcceptors']
# Inspect LD50 and NumHAcceptors one at a time.
for column in main_columns:
    # Print a heading for the lowest values.
    print(f'Five lowest {column} values:')
    # Display the five rows with the lowest values.
    display(clean_df.nsmallest(5, column)[inspection_columns])
    # Print a heading for the highest values.
    print(f'Five highest {column} values:')
    # Display the five rows with the highest values.
    display(clean_df.nlargest(5, column)[inspection_columns])

### 10. Save the cleaned dataset

Save all columns without a CSV index. Resolve the output directory from the loaded CSV path so this works from either the project root or `notebooks/`.

In [ ]:
# Set the cleaned CSV path inside data/processed.
processed_path = csv_path.parent.parent / 'processed' / 'ld50_cleaned.csv'
# Create the output folder if it does not already exist.
processed_path.parent.mkdir(parents=True, exist_ok=True)
# Save the cleaned data without the row index.
clean_df.to_csv(processed_path, index=False)
# Print the number of saved rows and the file location.
print(f'Saved {len(clean_df):,} rows to {processed_path.as_posix()}')
# Print the total rows removed and the reasons.
print(f'Total rows removed: {initial_rows - len(clean_df)} '
      f'({missing_rows_removed} missing target values; {exact_rows_removed} exact duplicates)')

### Cleaning summary

For the current raw dataset, 7,397 rows became **7,388 rows**: **1 row** was removed for missing `LD50` (none were missing `NumHAcceptors`), followed by **8 exact duplicate rows**, for **9 rows removed in total**. There were 55 repeated SMILES values before exact deduplication; the remaining 47 are retained as potentially separate toxicity records.

Both target columns are numeric and finite, with no missing values; all `NumHAcceptors` values are non-negative whole numbers. Observed `LD50` values range from **0.291 to 7.207**, and `NumHAcceptors` from **0 to 35**. Extreme values were inspected and retained, and `LD50` was not transformed.

The final dataset is saved to `data/processed/ld50_cleaned.csv`. The original `df` and raw CSV are unchanged.

## Dataset audit before machine learning

This read-only check reloads both CSVs and reproduces the existing cleaning rules. The tables below are calculated from the files; no analysis or data is changed.

- **7,396** is the number of raw records with observed `LD50`, also the intermediate count before exact deduplication. Historical notebook output (`5776a92`, `df.info()`/`df.describe()`) shows this non-missing count. The processed CSV already contained **7,388** rows when first committed (`edaed04`); 7,396 is not its final saved size.
- **7,388** is the cleaned record count and the Spearman sample. Removing one missing `LD50` and eight whole-row duplicates explains the entire reduction from 7,397. Spearman excludes no further rows.
- **7,378** is the complete-case sample selected in `src/data_visualisation.py` for `LD50`, `NumHAcceptors`, `MolWt`, and `MolLogP`: five rows lack `MolWt` and five different rows lack `MolLogP`. Both the HBA-only and adjusted models intentionally use this same sample. All ten records remain in the processed CSV. Adding `NumHDonors`, `RingCount`, and `HeavyAtomCount` causes no further loss.
- Exact comparison confirms every column and row matches the documented cleaning. Every discarded duplicate has a retained identical copy; no unique record with observed `LD50` and `NumHAcceptors` was lost. Repeated SMILES and extreme values were retained. Numeric/acceptor checks validate values without filtering rows.

**Starting point for future machine learning:** use all **7,388 records and 28 columns** in `data/processed/ld50_cleaned.csv`. The 7,378-record sample is appropriate only for a complete-case analysis of the variables above; decide predictor-specific missing-value handling later. These are record counts, not counts of unique compounds. No machine learning is run here.


In [ ]:
def audit_sample_sizes():
    from pathlib import Path
    import numpy as np
    import pandas as pd

    raw_path = Path('data/raw/ld50-smiles-descriptors-dataset.csv')
    if not raw_path.is_file():
        raw_path = Path('../data/raw/ld50-smiles-descriptors-dataset.csv')
    processed_path = raw_path.parent.parent / 'processed' / 'ld50_cleaned.csv'
    raw = pd.read_csv(raw_path)
    processed = pd.read_csv(processed_path)
    required = ['LD50', 'NumHAcceptors']
    model_columns = required + ['MolWt', 'MolLogP']
    audit_columns = model_columns + ['NumHDonors', 'RingCount', 'HeavyAtomCount']

    # Reproduce only the two documented removal steps, preserving raw indices.
    observed = raw.dropna(subset=required)
    duplicate_mask = observed.duplicated()
    expected = observed.drop_duplicates()
    pd.testing.assert_frame_equal(
        expected.reset_index(drop=True), processed, check_exact=True
    )
    # Confirm each discarded duplicate still has an identical representative.
    for index in observed.index[duplicate_mask]:
        row = observed.loc[index]
        matches = expected.eq(row).fillna(False) | (expected.isna() & row.isna())
        assert matches.all(axis=1).any(), f'No retained copy for raw index {index}'

    # Match the existing plotting script's selection, without fitting a model.
    model_data = processed[model_columns].apply(pd.to_numeric, errors='raise').dropna()
    spearman_data = processed.dropna(subset=required)
    complete_seven = processed.dropna(subset=audit_columns)
    numeric = processed[audit_columns].apply(pd.to_numeric, errors='raise')
    assert np.isfinite(numeric.fillna(0).to_numpy()).all()
    assert processed['NumHAcceptors'].ge(0).all()
    assert processed['NumHAcceptors'].mod(1).eq(0).all()

    steps = pd.DataFrame([
        ('Raw CSV', len(raw), 0),
        ('Remove missing LD50 / NumHAcceptors', len(observed), len(raw) - len(observed)),
        ('Remove exact duplicates = processed CSV / Spearman', len(expected),
         len(observed) - len(expected)),
        ('Select complete existing-model sample', len(model_data),
         len(processed) - len(model_data)),
    ], columns=['Step', 'Rows remaining', 'Rows removed at this step'])
    print(steps.to_string(index=False))
    print('\nMissing values in the requested variables:')
    print(pd.DataFrame({
        'Raw': raw[audit_columns].isna().sum(),
        'Processed': processed[audit_columns].isna().sum(),
    }).to_string())

    print('\nExcluded row identifiers (zero-based raw data indices, not CSV line numbers):')
    print('Missing required values:', raw.index[~raw.index.isin(observed.index)].tolist())
    print('Exact duplicate copies:', observed.index[duplicate_mask].tolist())
    for column in ['MolWt', 'MolLogP']:
        print(f'Missing {column}, retained in processed CSV:',
              expected.index[expected[column].isna()].tolist())
    overlap = (processed['MolWt'].isna() & processed['MolLogP'].isna()).sum()
    print(f'Missing both MolWt and MolLogP: {overlap}')
    print(f'Spearman: {len(spearman_data):,}; complete on all seven: {len(complete_seven):,}')

    smiles_counts = processed['SMILES'].value_counts(dropna=False)
    print(f'\nExact duplicate copies, raw -> processed: '
          f'{raw.duplicated().sum()} -> {processed.duplicated().sum()}')
    print(f'Repeated SMILES entries beyond the first, raw -> processed: '
          f'{raw["SMILES"].duplicated().sum()} -> {processed["SMILES"].duplicated().sum()}')
    print(f'Processed: {processed["SMILES"].nunique(dropna=False):,} distinct SMILES; '
          f'{smiles_counts.gt(1).sum()} repeated-SMILES groups contain '
          f'{smiles_counts[smiles_counts.gt(1)].sum()} retained records.')
    print(f'Exact CSV match: PASS ({processed.shape[0]:,} rows, {processed.shape[1]} columns).')
    print('All observed audit values are numeric and finite; acceptor counts are valid.')
    print('No additional rows removed; every discarded exact duplicate has a retained copy.')


audit_sample_sizes()


                                              Step  Rows remaining  Rows removed at this step
                                           Raw CSV            7397                          0
               Remove missing LD50 / NumHAcceptors            7396                          1
Remove exact duplicates = processed CSV / Spearman            7388                          8
             Select complete existing-model sample            7378                         10

Missing values in the requested variables:
                Raw  Processed
LD50              1          0
NumHAcceptors     0          0
MolWt             5          5
MolLogP           5          5
NumHDonors        0          0
RingCount         0          0
HeavyAtomCount    0          0

Excluded row identifiers (zero-based raw data indices, not CSV line numbers):
Missing required values: [381]
Exact duplicate copies: [4334, 4396, 5558, 6052, 6585, 7125, 7191, 7234]
Missing MolWt, retained in processed CSV: [155, 225, 25

## Initial statistical analysis: Spearman's rank correlation

Our main statistical test uses the cleaned dataframe, `clean_df`, to compare `NumHAcceptors` with `LD50`. The default test is two-sided.

- Spearman's rho measures the direction and strength of the association, from -1 to +1.
- A positive rho means more hydrogen-bond acceptors are associated with higher transformed LD50 values, therefore greater acute toxicity.
- A negative rho means the opposite: more acceptors are associated with lower transformed LD50 values and lower acute toxicity.
- A p-value below 0.05 means the association is statistically significant.
- Statistical significance does not necessarily mean the relationship is strong.

In [20]:
from scipy.stats import spearmanr

rho, p_value = spearmanr(clean_df['NumHAcceptors'], clean_df['LD50'])

print(f"Spearman rho: {rho:.3f}")
print(f"p-value: {p_value:.3e}")

sample_size = len(clean_df)
print(f"Sample size: {sample_size:,}")

Spearman rho: 0.252
p-value: 1.450e-107
Sample size: 7,388


In [21]:
from IPython.display import Markdown, display

# These strength labels are a rough descriptive guide, not strict cut-offs.
strength = "weak" if abs(rho) < 0.3 else "moderate" if abs(rho) < 0.7 else "strong"
direction = "positive" if rho > 0 else "negative" if rho < 0 else "zero"
significance = "statistically significant" if p_value < 0.05 else "not statistically significant"

display(Markdown(
    f"**Interpretation:** In **{sample_size:,}** cleaned records, "
    f"Spearman's rho is **{rho:.3f}** (p = **{p_value:.3e}**). "
    f"This is a **{strength} {direction}** association and is "
    f"**{significance}** at the 0.05 level. "
    "Judge the strength from rho, not the p-value alone."
))

**Interpretation:** In **7,388** cleaned records, Spearman's rho is **0.252** (p = **1.450e-107**). This is a **weak positive** association and is **statistically significant** at the 0.05 level. Judge the strength from rho, not the p-value alone.